In [2]:
%pip install pandas numpy scikit-learn imbalanced-learn 

In [53]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler

from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier


from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [54]:
df = pd.read_csv("cumulative.csv")

print(df.head())
print(df.shape)

   rowid     kepid kepoi_name   kepler_name koi_disposition koi_pdisposition  \
0      1  10797460  K00752.01  Kepler-227 b       CONFIRMED        CANDIDATE   
1      2  10797460  K00752.02  Kepler-227 c       CONFIRMED        CANDIDATE   
2      3  10811496  K00753.01           NaN  FALSE POSITIVE   FALSE POSITIVE   
3      4  10848459  K00754.01           NaN  FALSE POSITIVE   FALSE POSITIVE   
4      5  10854555  K00755.01  Kepler-664 b       CONFIRMED        CANDIDATE   

   koi_score  koi_fpflag_nt  koi_fpflag_ss  koi_fpflag_co  ...  \
0      1.000              0              0              0  ...   
1      0.969              0              0              0  ...   
2      0.000              0              1              0  ...   
3      0.000              0              1              0  ...   
4      1.000              0              0              0  ...   

   koi_steff_err2  koi_slogg  koi_slogg_err1  koi_slogg_err2  koi_srad  \
0           -81.0      4.467           0.064    

In [55]:
print(df.isnull().sum())

rowid                   0
kepid                   0
kepoi_name              0
kepler_name          7270
koi_disposition         0
koi_pdisposition        0
koi_score            1510
koi_fpflag_nt           0
koi_fpflag_ss           0
koi_fpflag_co           0
koi_fpflag_ec           0
koi_period              0
koi_period_err1       454
koi_period_err2       454
koi_time0bk             0
koi_time0bk_err1      454
koi_time0bk_err2      454
koi_impact            363
koi_impact_err1       454
koi_impact_err2       454
koi_duration            0
koi_duration_err1     454
koi_duration_err2     454
koi_depth             363
koi_depth_err1        454
koi_depth_err2        454
koi_prad              363
koi_prad_err1         363
koi_prad_err2         363
koi_teq               363
koi_teq_err1         9564
koi_teq_err2         9564
koi_insol             321
koi_insol_err1        321
koi_insol_err2        321
koi_model_snr         363
koi_tce_plnt_num      346
koi_tce_delivname     346
koi_steff   

In [56]:
drop_cols = [
    'kepid',
    'kepoi_name',
    'kepler_name'
]

existing = [col for col in drop_cols if col in df.columns]

df.drop(columns=existing, inplace=True)




In [57]:
target = "koi_disposition"

X = df.drop(target, axis=1)

y = df[target]

In [58]:
# 1. Convert text columns to numbers first
le = LabelEncoder()
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = le.fit_transform(X[col].astype(str))

# 2. DROP COMPLETELY EMPTY COLUMNS (Add this line!)
X = X.dropna(how='all', axis=1)

# 3. Now the median imputer and DataFrame creation will work flawlessly
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

In [59]:
print(y.value_counts())

koi_disposition
FALSE POSITIVE    5023
CONFIRMED         2293
CANDIDATE         2248
Name: count, dtype: int64


In [60]:
smote = SMOTE(random_state=42)

X_resampled, y_resampled = smote.fit_resample(X, y)

In [61]:
print(y_resampled.value_counts())

koi_disposition
CONFIRMED         5023
FALSE POSITIVE    5023
CANDIDATE         5023
Name: count, dtype: int64


In [62]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled,
    y_resampled,
    test_size=0.2,
    random_state=42,
    stratify=y_resampled
)

In [63]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [64]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)  # <-- Changed X_train to X_train_scaled

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [65]:
lr_pred = lr_model.predict(X_test_scaled)

In [66]:
print("LOGISTIC REGRESSION")

print("Accuracy:",
      accuracy_score(y_test, lr_pred))

print("Precision:",
      precision_score(y_test, lr_pred,
                      average='weighted'))

print("Recall:",
      recall_score(y_test, lr_pred,
                   average='weighted'))

print("F1 Score:",
      f1_score(y_test, lr_pred,
               average='weighted'))

LOGISTIC REGRESSION
Accuracy: 0.8908427339084274
Precision: 0.8924957016807922
Recall: 0.8908427339084274
F1 Score: 0.8907671074471426


In [67]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

,n_estimators,200
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [68]:
rf_pred = rf_model.predict(X_test)

In [69]:
print("\nRANDOM FOREST")

print("Accuracy:",
      accuracy_score(y_test, rf_pred))

print("Precision:",
      precision_score(y_test, rf_pred,
                      average='weighted'))

print("Recall:",
      recall_score(y_test, rf_pred,
                   average='weighted'))

print("F1 Score:",
      f1_score(y_test, rf_pred,
               average='weighted'))


RANDOM FOREST
Accuracy: 0.9386197743861977
Precision: 0.938545085193996
Recall: 0.9386197743861977
F1 Score: 0.9385622364704378


In [70]:
knn_model = KNeighborsClassifier(
    n_neighbors=5
)

knn_model.fit(X_train, y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [71]:
knn_pred = knn_model.predict(X_test)

In [72]:
print("KNN")

print("Accuracy:",
      accuracy_score(y_test, knn_pred))

print("Precision:",
      precision_score(y_test, knn_pred,
                      average='weighted'))

print("Recall:",
      recall_score(y_test, knn_pred,
                   average='weighted'))

print("F1 Score:",
      f1_score(y_test, knn_pred,
               average='weighted'))

KNN
Accuracy: 0.7325812873258128
Precision: 0.7416513148925362
Recall: 0.7325812873258128
F1 Score: 0.7319368901129701
